In [1]:
from nemo.collections.asr.parts.submodules.wfst_decoder import RivaGpuWfstDecoder

from inference_funcs import load_bit_phoneme_model, load_gru, evaluate_model
from dataset import getDatasetLoaders
import numpy as np
import torch
import torch.nn.functional as F

language_model_fst_path = "/data/code/nejm-brain-to-text/language_model/pretrained_language_models/openwebtext_1gram_lm_sil/TLG_opt_with_symbols.fst"
language_model_path_3g = '/data/lm/TLG_opt_with_symbols.fst'

max_mem = 500000000
blank_penalty = 0.7
lm_weight = 1.0
beam_size = 18
nbest_size = 1

decoder = RivaGpuWfstDecoder(lm_fst=language_model_path_3g, decoding_mode="nbest", 
                             beam_size=beam_size, lm_weight=lm_weight,
                             nbest_size=nbest_size, max_mem=max_mem, blank_penalty=blank_penalty)


load_predictions = True

device = 'cuda'

model_filepath = "/data/models/transformer_short_training_fixed_seed_0/"
gru = False

if load_predictions:
    
    log_probs_arranged = torch.load(f"{model_filepath}log_probs_arranged.pth").to(dtype=torch.float32, device=device)
    log_probs_length = torch.load(f"{model_filepath}log_probs_length.pth").to(dtype=torch.int64, device='cpu')

else: 
    
    if gru:
        model, args = load_gru(model_filepath)
        
    else:    
        model, args = load_bit_phoneme_model(model_filepath)
        
    model = model.to(device)

    data_file = '/data/neural_data/ptDecoder_ctc_both'
    trainLoaders, testLoaders, loadedData = getDatasetLoaders(
            data_file, 8, None, 
            False
        )

    outputs, cer, per_day_cer = evaluate_model(model, loadedData, args, partition='test', device='cuda', verbose=False)

    num_classes = 41
    if gru:
        add_length = 1
    else:
        add_length = 0

    print(add_length)
    logits = np.zeros((len(outputs['logits']), max(outputs['logitLengths'])+add_length, num_classes))
    for idx, l in enumerate(outputs['logits']):
        l_length = outputs['logitLengths'][idx]
        if gru:
            l_length += add_length
        logits[idx, :l_length, :] = l
        
    logits_torch = torch.from_numpy(logits)
    log_probs = F.log_softmax(logits_torch, dim=-1).to(dtype=torch.float32, device=device)
    log_probs_length = torch.from_numpy(np.array(outputs['logitLengths'])).to(dtype=torch.int64, device='cpu')

    log_probs_blank_last = torch.concat((log_probs[:, :, 1:], log_probs[:, :, 0:1]), dim=-1) # move blank to end
    log_probs_arranged = torch.concat((log_probs_blank_last[:, :, -1:], log_probs_blank_last[:, :, -2:-1], log_probs_blank_last[:, :, :-2]), dim=-1)

    torch.save(log_probs_arranged, f"{model_filepath}log_probs_arranged.pth")
    torch.save(log_probs_length, f"{model_filepath}log_probs_length.pth")

  

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import pickle
with open('/data/text/validation_sentences_ground_truth.pkl', 'rb') as f:
    val_ground_truth_all = pickle.load(f)

In [3]:
log_probs_one_sample = torch.unsqueeze(log_probs_arranged[400], dim=0)
log_probs_length_one_sample = torch.unsqueeze(log_probs_length[400], dim=0)
print(log_probs_length_one_sample)
print(log_probs_one_sample.shape)
print(log_probs_length_one_sample)

tensor([49])
torch.Size([1, 184, 41])
tensor([49])


In [4]:
hypotheses = decoder._decode_nbest(log_probs_arranged, log_probs_length)

WARNING ([5.5]:ComputeH2HCopies():cudadecoder/cuda-decoder.cc:2083) Resized host vector containing 'acoustic costs'. This can cause latency spikes in production. Please use smaller input audio sequences or increase ntokens_pre_allocatedOld capacity: 1000000, New capacity:1999006
WARNING ([5.5]:ComputeH2HCopies():cudadecoder/cuda-decoder.cc:2100) Resized host vector containing 'infotoken'. This can cause latency spikes in production. Please use smaller input audio sequences or increase ntokens_pre_allocatedOld capacity: 1000000, New capacity:1999006
WARNING ([5.5]:ComputeH2HCopies():cudadecoder/cuda-decoder.cc:2083) Resized host vector containing 'acoustic costs'. This can cause latency spikes in production. Please use smaller input audio sequences or increase ntokens_pre_allocatedOld capacity: 1000000, New capacity:1986474
WARNING ([5.5]:ComputeH2HCopies():cudadecoder/cuda-decoder.cc:2100) Resized host vector containing 'infotoken'. This can cause latency spikes in production. Please u

In [15]:
decoded_sentences = []
for i in range(880):
    words_tuple = hypotheses[i]._hypotheses[0].words
    decoded_sentences.append(' '.join(words_tuple).lower())

In [23]:
from cer_wer import  _cer_and_wer
_, wer, _ =  _cer_and_wer(decodedSentences=decoded_sentences, trueSentences=val_ground_truth_all)
print(wer)

0.23822513184215313
